In [ ]:
%pip install pandas kaggle trl liger-kernel accelerate unsloth
# !kaggle competitions download -c deep-past-initiative-machine-translation
# !unzip deep-past-initiative-machine-translation.zip -d deep_past_data_past_data

import pandas as pd
train_df = pd.read_csv("deep_past_data_past_data/train.csv")
test_df = pd.read_csv("deep_past_data_past_data/test.csv")
definitions_df = pd.read_csv("deep_past_data_past_data/eBL_Dictionary.csv")

In [ ]:
train_df

In [ ]:
definitions_df

In [ ]:
import re
in_quotes = r'"([^"]*)"' # regex to capture text within quotes

definitions_df["translation"] = definitions_df["definition"].fillna("").apply(
    lambda x: re.findall(in_quotes, x)[0] if re.findall(in_quotes, x) else ""
)
definitions_df["transliteration"] = definitions_df["word"]
definitions_df = definitions_df[["transliteration", "translation"]]
definitions_df

In [ ]:
def create_text_column(row):
    if "translation" not in row or pd.isna(row["translation"]):
        return "Akkadian:" + row["transliteration"] + ":" + "English:"
    return "Akkadian:" + row["transliteration"] + ":" + "English: " + row["translation"]

In [ ]:
train_df["text"] = train_df.apply(create_text_column, axis=1)
test_df["text"] = test_df.apply(create_text_column, axis=1)
definitions_df["text"] = definitions_df.apply(create_text_column, axis=1)

validation_df = train_df.sample(frac=0.1, random_state=0)
train_df = train_df.drop(validation_df.index)

from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
validation_dataset = Dataset.from_pandas(validation_df)
test_dataset = Dataset.from_pandas(test_df)
definitions_dataset = Dataset.from_pandas(definitions_df)

In [ ]:
from unsloth import FastLanguageModel
from trl.trainer.sft_trainer import SFTTrainer
from trl.trainer.sft_config import SFTConfig
import torch

# Safe in spawned processes; will crash if the launcher uses fork.
bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen3-4B",
    dtype = torch.bfloat16 if bf16_ok else torch.float16,
    trust_remote_code = True,
    max_seq_length = 1024,
    load_in_4bit = True,
    full_finetuning = False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32
    , # This controls the rank of the LoRA update. Increase for larger models/datasets for better quality.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 512, # This controls the scaling of the LoRA update. Usually set to r*16.
    lora_dropout = 0.0, # Dropout for LoRA layers.
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 8888,
    use_rslora = True,
    loftq_config = None,
)

# First train on definitions to give model basic vocabulary
definitions_training_args = SFTConfig(
    output_dir="akkadian-definitions-model-dictionary",
    dataset_text_field="text",
    report_to = "none", # Use TrackIO/WandB etc
    num_train_epochs=5,
    per_device_train_batch_size=2048,
    # load_best_model_at_end=True,
    # batch_eval_metrics=True,
    # eval_strategy="epoch",
    # save_strategy="epoch",
)

definitions_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=definitions_dataset,
    # eval_dataset=validation_dataset,
    args=definitions_training_args,
)
definitions_trainer.train()
definitions_trainer.save_model(definitions_training_args.output_dir)

In [ ]:
# Safe in spawned processes; will crash if the launcher uses fork.
bf16_ok = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = definitions_training_args.output_dir,
    dtype = torch.bfloat16 if bf16_ok else torch.float16,
    trust_remote_code = True,
    max_seq_length = 1024,
    load_in_4bit = True,
    full_finetuning = False
)

training_args = SFTConfig(
    output_dir="akkadian-translation-model",
    dataset_text_field="text",
    report_to = "none", # Use TrackIO/WandB etc
    num_train_epochs=3,
    per_device_train_batch_size=1024,
    load_best_model_at_end=True,
    batch_eval_metrics=True,
    eval_strategy="epoch",
    save_strategy="epoch",
)

# Then train on main translation dataset
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    args=training_args,
)
trainer.train()
trainer.save_model(training_args.output_dir)

In [ ]:
# Load the model and tokenizer using Unsloth for inference
model_path = "akkadian-translation-model"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = 512,
    dtype = None, # Auto-detect
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# Function to generate translation from the text column using the optimized model
def generate_translation(text):
    inputs = tokenizer([text], return_tensors = "pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens = 100, use_cache = True)
    # Decode only the generated part by skipping the prompt tokens
    generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return generated_text.strip()

# Generate predictions
# We use the 'text' field which contains the prompt format defined in create_text_column
debug_df = train_df.head(10).copy()

def remove_translation_part(text):
    # Remove the "English: " part to only keep the Akkadian transliteration
    if "English:" in text:
        return text.split("English:")[0] + "English:"
    return text

debug_df["text"] = debug_df["text"].apply(remove_translation_part)

debug_df["predicted_translation"] = debug_df["text"].apply(generate_translation)
debug_df["id"] = debug_df.index

# Save result
submission_df = debug_df[["id", "predicted_translation"]]
submission_df.to_csv("akkadian_translations.csv", index=False)

out = debug_df[["transliteration", "translation", "predicted_translation"]]
out.to_csv("akkadian_debug_output.csv", index=False)

out